# MarketPulse LK – Advanced Statistical Analysis
## Advanced Statistics for Data Science
**Project Title**: Statistical Modelling of Volatility, Financial Risk and Market Regime Changes in Selected Colombo Stock Exchange Equities Using Time-Series and Machine Learning Techniques  
**Equities under Investigation**:
1. **John Keells Holdings PLC (JKH)** – Diversified Conglomerate / Market Bellwether
2. **Hayleys PLC (HAYL)** – Export, Logistics & Industrial Manufacturing Conglomerate
3. **Vallibel One PLC (VONE)** – Industrial Manufacturing & Financial Services Conglomerate

---

### Analytical Architecture & Research Scope (Part 05)
This notebook implements **Part 05: Advanced Statistical Analysis**. Building on the exploratory empirical baseline established in Part 04 (EDA), this module executes formal asymptotic and non-parametric hypothesis testing to establish the core time-series and distributional properties of Colombo Stock Exchange daily equity returns.

### Core Objectives:
1. **Stationarity & Integration Analysis**: Augmented Dickey-Fuller (ADF) and Kwiatkowski-Phillips-Schmidt-Shin (KPSS) tests to ascertain the integration order $I(d)$ of price and return series.
2. **Autocorrelation & Serial Dependence**: Autocorrelation (ACF) and Partial Autocorrelation (PACF) up to lag 40 for levels, squared returns ($r_t^2$), and absolute returns ($|r_t|$).
3. **Portmanteau Autocorrelation Tests**: Ljung-Box $Q$-tests at multiple lag orders ($m = 10, 20$) testing for weak-form efficiency and linear/non-linear dependence.
4. **Normality Goodness-of-Fit Synthesis**: Consolidating empirical higher moments with Jarque-Bera tests from EDA.
5. **Conditional Heteroskedasticity (ARCH-LM)**: Engle's Lagrange Multiplier test to verify autoregressive conditional heteroskedasticity and establish statistical justification for GARCH modeling.
6. **Parameter Instability & Structural Breaks**: OLS-CUSUM fluctuation tests and Zivot-Andrews endogenous single-break unit root tests.
7. **Asymmetric Volatility Dynamics**: Engle-Ng sign and size bias tests evaluating leverage and asymmetric information shocks.
8. **Comparative Synthesis**: Cross-sectional econometric comparison across JKH, HAYL, and VONE.


In [2]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf, zivot_andrews
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch, breaks_cusumolsresid
import arch
from arch import arch_model

# Visual aesthetics configuration for academic reporting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['figure.titlesize'] = 13
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 10

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# Output directory routing
figures_base = os.path.join('..', 'outputs', 'figures', 'advanced_statistics')
tables_base = os.path.join('..', 'outputs', 'tables')
processed_dir = os.path.join('..', 'data', 'processed')

os.makedirs(figures_base, exist_ok=True)
os.makedirs(tables_base, exist_ok=True)

# Color scheme matching project standard
palette = {'JKH': '#1f77b4', 'HAYL': '#ff7f0e', 'VONE': '#2ca02c'}

print(f"Environment Verification:")
print(f"- Python: {sys.version.split()[0]} | Pandas: {pd.__version__} | NumPy: {np.__version__}")
print(f"- Statsmodels: {sm.__version__} | ARCH: {arch.__version__}")


ModuleNotFoundError: No module named 'numpy'

---
## STEP 01 – Dataset Loading and Verification

### 1.1 Purpose of the Analysis
Prior to conducting formal asymptotic hypothesis tests, this step verifies:
- Accessibility and schema conformance of cleaned datasets in `data/processed/`.
- Date range synchronization across 2014-01-02 to 2026-10-14.
- Availability and definition of price metrics (`Adjusted_Close`) and return series:
  - Percentage return: $R_t = \left(\frac{P_t}{P_{t-1}} - 1\right) \times 100$
  - Continuously compounded log return: $r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$
- Cross-referencing previously completed EDA tables in `outputs/tables/` to avoid redundant computation.


In [ ]:
equities = ['JKH', 'HAYL', 'VONE']
dfs = {}

for eq in equities:
    filepath = os.path.join(processed_dir, f"{eq}_cleaned.csv")
    df = pd.read_csv(filepath)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Calculate exact percentage and log returns from Adjusted_Close
    df['Pct_Return'] = df['Adjusted_Close'].pct_change() * 100
    df['Log_Return_Calc'] = np.log(df['Adjusted_Close'] / df['Adjusted_Close'].shift(1))
    
    # Preserve row 0 values
    if pd.notnull(df.loc[0, 'Log_Return']):
        df.loc[0, 'Log_Return_Calc'] = df.loc[0, 'Log_Return']
    if pd.notnull(df.loc[0, 'Change_%']):
        df.loc[0, 'Pct_Return'] = df.loc[0, 'Change_%']
        
    dfs[eq] = df

# Generate Verification Summary Table
verif_rows = []
for eq in equities:
    df = dfs[eq]
    verif_rows.append({
        'Company': eq,
        'Observations (T)': len(df),
        'Start Date': df['Date'].min().strftime('%Y-%m-%d'),
        'End Date': df['Date'].max().strftime('%Y-%m-%d'),
        'Has Adjusted_Close': 'Adjusted_Close' in df.columns,
        'Has Pct_Return': 'Pct_Return' in df.columns,
        'Has Log_Return': 'Log_Return_Calc' in df.columns,
        'Pct_Return Nulls': df['Pct_Return'].isnull().sum(),
        'Log_Return Nulls': df['Log_Return_Calc'].isnull().sum(),
        'Monotonic Ascending': df['Date'].is_monotonic_increasing
    })

verif_table = pd.DataFrame(verif_rows)
display(verif_table)
print(verif_table.to_string(index=False))


,Company,Observations (T),Start Date,End Date,Has Adjusted_Close,Has Pct_Return,Has Log_Return,Pct_Return Nulls,Log_Return Nulls,Monotonic Ascending
0,JKH,2971,2014-01-02,2026-10-14,True,True,True,0,0,True
1,HAYL,2725,2014-01-02,2026-10-14,True,True,True,0,0,True
2,VONE,2910,2014-01-02,2026-10-14,True,True,True,0,0,True


Company  Observations (T) Start Date   End Date  Has Adjusted_Close  Has Pct_Return  Has Log_Return  Pct_Return Nulls  Log_Return Nulls  Monotonic Ascending
    JKH              2971 2014-01-02 2026-10-14                True            True            True                 0                 0                 True
   HAYL              2725 2014-01-02 2026-10-14                True            True            True                 0                 0                 True
   VONE              2910 2014-01-02 2026-10-14                True            True            True                 0                 0                 True
